# Pydantic to structure Gemini outputs

In [1]:
# from dotenv import load_dotenv
# import os 
from google import genai

# load_dotenv()

client = genai.Client()

response = client.models.generate_content(
    model = "gemini-2.5-flash", contents="Tell me a programming joke"
)

response



GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""Why do programmers prefer dark mode?

Because light attracts bugs!

---

Or, a classic:

There are 10 types of people in the world: those who understand binary, and those who don't."""
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='mJ6-aKLpCoXlvdIP2di-gQ4',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=46,
    prompt_token_count=6,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=6
      ),
    ],
    thoughts_token_count=271,
    total_token_count=323
  )
)

In [2]:
response.text

"Why do programmers prefer dark mode?\n\nBecause light attracts bugs!\n\n---\n\nOr, a classic:\n\nThere are 10 types of people in the world: those who understand binary, and those who don't."

In [3]:
print(response.text)

Why do programmers prefer dark mode?

Because light attracts bugs!

---

Or, a classic:

There are 10 types of people in the world: those who understand binary, and those who don't.


In [4]:
def ask_llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

ask_llm("Du är en Göteborgare, ge mig ett skämt som är go")

'Hallå där, goa gubbe/gumma! Kul att du hör av dig! Ett göteborgsskämt ska du få, det är la go!\n\n***\n\nDet var två gubbar som satt på en bänk nere vid kajen och kikade ut över vattnet.\n\nSäger den ena: "Fasen, Kalle, det e la gôtt me lite lugn och ro här vid vattnet!"\n\nKalle svarar: "Ja, det e det la. Men du, säg mig, om du skulle ramla i vattnet nu, skulle du kunna simma till land?"\n\nGubben funderar en stund, kliar sig i skägget och säger sen: "Näe... Men jag e la så pass stark att jag skulle kunna **gå på botten**!"\n\n***\n\nHaha! Go e den, inte sant? Ha en alldeles strålande dag nu!'

## Try to get data from our LLM

In [5]:
response = ask_llm("""
    Du är en expert inom köp och sälj av bostäder, likt en proffsig mäklare.
    Generera bostadspriser, månadsavgifter, address, stad, boarea i jsonformat (ej markdown)

    Exempel:
            {
                "address": "Fågelvägen 5,
                "price_sek": 3000000,
                "city": "Göteborg",
                "monthly_fee": 4000,
                "area": 60
            }   
                   
    Ge mig en lista på 5 bostäder
""")

response

'[\n    {\n        "address": "Vasagatan 12, Lgh 301",\n        "price_sek": 4850000,\n        "city": "Stockholm",\n        "monthly_fee": 3800,\n        "area": 75\n    },\n    {\n        "address": "Kungsladugårdsgatan 25B",\n        "price_sek": 3200000,\n        "city": "Göteborg",\n        "monthly_fee": 4500,\n        "area": 62\n    },\n    {\n        "address": "Södra Förstadsgatan 88, Vån 4",\n        "price_sek": 2750000,\n        "city": "Malmö",\n        "monthly_fee": 2900,\n        "area": 50\n    },\n    {\n        "address": "Ulltunalundsvägen 15B",\n        "price_sek": 3950000,\n        "city": "Uppsala",\n        "monthly_fee": 5100,\n        "area": 88\n    },\n    {\n        "address": "Linnégatan 4, Ö.g.",\n        "price_sek": 1995000,\n        "city": "Linköping",\n        "monthly_fee": 3400,\n        "area": 45\n    }\n]'

In [6]:
print(response)

[
    {
        "address": "Vasagatan 12, Lgh 301",
        "price_sek": 4850000,
        "city": "Stockholm",
        "monthly_fee": 3800,
        "area": 75
    },
    {
        "address": "Kungsladugårdsgatan 25B",
        "price_sek": 3200000,
        "city": "Göteborg",
        "monthly_fee": 4500,
        "area": 62
    },
    {
        "address": "Södra Förstadsgatan 88, Vån 4",
        "price_sek": 2750000,
        "city": "Malmö",
        "monthly_fee": 2900,
        "area": 50
    },
    {
        "address": "Ulltunalundsvägen 15B",
        "price_sek": 3950000,
        "city": "Uppsala",
        "monthly_fee": 5100,
        "area": 88
    },
    {
        "address": "Linnégatan 4, Ö.g.",
        "price_sek": 1995000,
        "city": "Linköping",
        "monthly_fee": 3400,
        "area": 45
    }
]


## Parse and validate data

In [7]:
from pydantic import BaseModel, Field
import json 

class Apartment(BaseModel):
    address: str 
    city: str 
    price_sek: int = Field(gt=1000000, lt = 8000000) 
    monthly_fee: int 
    area: int 

class ApartmentList(BaseModel):
    objects: list[Apartment]


apartments = ApartmentList.model_validate({"objects": json.loads(response)})
apartments
    

ApartmentList(objects=[Apartment(address='Vasagatan 12, Lgh 301', city='Stockholm', price_sek=4850000, monthly_fee=3800, area=75), Apartment(address='Kungsladugårdsgatan 25B', city='Göteborg', price_sek=3200000, monthly_fee=4500, area=62), Apartment(address='Södra Förstadsgatan 88, Vån 4', city='Malmö', price_sek=2750000, monthly_fee=2900, area=50), Apartment(address='Ulltunalundsvägen 15B', city='Uppsala', price_sek=3950000, monthly_fee=5100, area=88), Apartment(address='Linnégatan 4, Ö.g.', city='Linköping', price_sek=1995000, monthly_fee=3400, area=45)])

In [8]:
apartments.objects

[Apartment(address='Vasagatan 12, Lgh 301', city='Stockholm', price_sek=4850000, monthly_fee=3800, area=75),
 Apartment(address='Kungsladugårdsgatan 25B', city='Göteborg', price_sek=3200000, monthly_fee=4500, area=62),
 Apartment(address='Södra Förstadsgatan 88, Vån 4', city='Malmö', price_sek=2750000, monthly_fee=2900, area=50),
 Apartment(address='Ulltunalundsvägen 15B', city='Uppsala', price_sek=3950000, monthly_fee=5100, area=88),
 Apartment(address='Linnégatan 4, Ö.g.', city='Linköping', price_sek=1995000, monthly_fee=3400, area=45)]

In [9]:
apartments.objects[1].address, apartments.objects[1].city

('Kungsladugårdsgatan 25B', 'Göteborg')

In [10]:
addresses = [apartment.address for apartment in apartments.objects ]

addresses

['Vasagatan 12, Lgh 301',
 'Kungsladugårdsgatan 25B',
 'Södra Förstadsgatan 88, Vån 4',
 'Ulltunalundsvägen 15B',
 'Linnégatan 4, Ö.g.']

In [11]:
addresses = [
    apartment.address
    for apartment in apartments.objects
    if apartment.price_sek < 4000000
]

addresses

['Kungsladugårdsgatan 25B',
 'Södra Förstadsgatan 88, Vån 4',
 'Ulltunalundsvägen 15B',
 'Linnégatan 4, Ö.g.']

Get address, city, price, monthly_fee for the interval 4M - 8M 

In [ ]:
filtered_adresses = [{"address":apt.address, "city":apt.city, "price_sek":apt.price_sek, "monthly_fee":apt.monthly_fee}  for apt in apartments.objects if apt.price_sek > 3000000 and apt.price_sek < 8000000]
filtered_adresses
# if 3000000 < apt.price_sek < 8000000

[{'address': 'Vasagatan 12, Lgh 301',
  'city': 'Stockholm',
  'price_sek': 4850000,
  'monthly_fee': 3800},
 {'address': 'Kungsladugårdsgatan 25B',
  'city': 'Göteborg',
  'price_sek': 3200000,
  'monthly_fee': 4500},
 {'address': 'Ulltunalundsvägen 15B',
  'city': 'Uppsala',
  'price_sek': 3950000,
  'monthly_fee': 5100}]

save in duckdb

In [17]:
import pandas as pd

df = pd.DataFrame(filtered_adresses)
df

,address,city,price_sek,monthly_fee
0,"Vasagatan 12, Lgh 301",Stockholm,4850000,3800
1,Kungsladugårdsgatan 25B,Göteborg,3200000,4500
2,Ulltunalundsvägen 15B,Uppsala,3950000,5100


In [22]:
import duckdb

with duckdb.connect("housing.duckdb") as con:
    con.execute("CREATE SCHEMA IF NOT EXISTS stg")
    con.execute("CREATE OR REPLACE TABLE stg.apartments AS SELECT * FROM df")
    new_df = con.query("SELECT * FROM stg.apartments").df()
new_df

,address,city,price_sek,monthly_fee
0,"Vasagatan 12, Lgh 301",Stockholm,4850000,3800
1,Kungsladugårdsgatan 25B,Göteborg,3200000,4500
2,Ulltunalundsvägen 15B,Uppsala,3950000,5100


In [23]:
homes = [home for home in apartments.objects if 3000000 < home.price_sek < 8000000]

df_homes = pd.DataFrame(home.model_dump(include={"address", "city", "price_sek", "monthly_fee"}) for home in homes)

df_homes

,address,city,price_sek,monthly_fee
0,"Vasagatan 12, Lgh 301",Stockholm,4850000,3800
1,Kungsladugårdsgatan 25B,Göteborg,3200000,4500
2,Ulltunalundsvägen 15B,Uppsala,3950000,5100


In [24]:
df_homes.to_csv("filtered_homes.csv", index=False)

In [25]:
apartments

ApartmentList(objects=[Apartment(address='Vasagatan 12, Lgh 301', city='Stockholm', price_sek=4850000, monthly_fee=3800, area=75), Apartment(address='Kungsladugårdsgatan 25B', city='Göteborg', price_sek=3200000, monthly_fee=4500, area=62), Apartment(address='Södra Förstadsgatan 88, Vån 4', city='Malmö', price_sek=2750000, monthly_fee=2900, area=50), Apartment(address='Ulltunalundsvägen 15B', city='Uppsala', price_sek=3950000, monthly_fee=5100, area=88), Apartment(address='Linnégatan 4, Ö.g.', city='Linköping', price_sek=1995000, monthly_fee=3400, area=45)])

In [29]:
apartments.model_dump()

{'objects': [{'address': 'Vasagatan 12, Lgh 301',
   'city': 'Stockholm',
   'price_sek': 4850000,
   'monthly_fee': 3800,
   'area': 75},
  {'address': 'Kungsladugårdsgatan 25B',
   'city': 'Göteborg',
   'price_sek': 3200000,
   'monthly_fee': 4500,
   'area': 62},
  {'address': 'Södra Förstadsgatan 88, Vån 4',
   'city': 'Malmö',
   'price_sek': 2750000,
   'monthly_fee': 2900,
   'area': 50},
  {'address': 'Ulltunalundsvägen 15B',
   'city': 'Uppsala',
   'price_sek': 3950000,
   'monthly_fee': 5100,
   'area': 88},
  {'address': 'Linnégatan 4, Ö.g.',
   'city': 'Linköping',
   'price_sek': 1995000,
   'monthly_fee': 3400,
   'area': 45}]}

In [32]:
apartments.model_dump_json()

'{"objects":[{"address":"Vasagatan 12, Lgh 301","city":"Stockholm","price_sek":4850000,"monthly_fee":3800,"area":75},{"address":"Kungsladugårdsgatan 25B","city":"Göteborg","price_sek":3200000,"monthly_fee":4500,"area":62},{"address":"Södra Förstadsgatan 88, Vån 4","city":"Malmö","price_sek":2750000,"monthly_fee":2900,"area":50},{"address":"Ulltunalundsvägen 15B","city":"Uppsala","price_sek":3950000,"monthly_fee":5100,"area":88},{"address":"Linnégatan 4, Ö.g.","city":"Linköping","price_sek":1995000,"monthly_fee":3400,"area":45}]}'

In [31]:
with open("apartments.json", "w", encoding="utf-8") as f:
    f.write(apartments.model_dump_json(indent=3))

In [8]:
df = pd.read_csv("filtered_homes.csv")
df

,address,city,price_sek,monthly_fee
0,"Vasagatan 12, Lgh 301",Stockholm,4850000,3800
1,Kungsladugårdsgatan 25B,Göteborg,3200000,4500
2,Ulltunalundsvägen 15B,Uppsala,3950000,5100


In [12]:
import dlt 
import duckdb
import pandas as pd

df = pd.read_csv("filtered_homes.csv")

@dlt.resource(write_disposition="replace", table_name="apartment")
def load_data():
    for d in df.to_dict(orient="records"):
        yield d

pipeline = dlt.pipeline(
    pipeline_name="apartments",
    destination="duckdb",
    dataset_name="staging"   
)

load_info = pipeline.run(load_data())
print(load_info)

Pipeline apartments load step completed in 0.14 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Users\organ\Repos\ml_ai\ai_engineering_pontus_agren_grundstrom\code-alongs\pydantic_basics\apartments.duckdb location to store data
Load package 1757331799.1031902 is LOADED and contains no failed jobs
